In [ ]:
!pip install -q openai-whisper
!apt-get update -qq && apt-get install -y -qq ffmpeg

In [ ]:
from pathlib import Path
import json
import shutil
import torch
import whisper
from IPython.display import display, Audio, FileLink

# Change this to your Kaggle dataset slug/folder name
AUDIO_DIR = Path("/kaggle/input/my-whisper-audio")

audio_files = [
    AUDIO_DIR / "aliens.mp3",
    AUDIO_DIR / "art.mp3",
]

output_root = Path("/kaggle/working/whisper_output")
output_root.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("base", device=device)

for audio_path in audio_files:
    if not audio_path.exists():
        print(f"Missing: {audio_path}")
        continue

    print(f"\nTranscribing: {audio_path.name}")
    display(Audio(str(audio_path)))

    audio_name = audio_path.stem
    lesson_folder = output_root / audio_name
    lesson_folder.mkdir(exist_ok=True)

    copied_audio = lesson_folder / audio_path.name
    shutil.copy2(audio_path, copied_audio)

    result = model.transcribe(
        str(audio_path),
        language="en",
        fp16=(device == "cuda")
    )

    timestamp_data = [
        {
            "start": segment["start"],
            "end": segment["end"],
            "text": segment["text"].strip()
        }
        for segment in result["segments"]
    ]

    (lesson_folder / f"raw_{audio_name}_timestamp.json").write_text(
        json.dumps(timestamp_data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    transcript = "\n".join(item["text"] for item in timestamp_data)
    (lesson_folder / f"raw_{audio_name}.txt").write_text(
        transcript,
        encoding="utf-8"
    )

    print(transcript)
    print(f"Saved to: {lesson_folder}")

zip_path = shutil.make_archive(
    "/kaggle/working/whisper_output",
    "zip",
    output_root
)

print("\nFinished.")
display(FileLink(zip_path))